# E0 — Build the PhageLysClass EDA universe

Build one exact-sequence-level CSV for the EDA.

The table keeps the frozen primary P/N/U/challenge assignment and also reconstructs
independent evidence flags from the UniProt sequence-level evidence so that overlaps such as
`N_easy + U_explicit` or `N_hard + U_other_unresolved` are preserved.

`U` is not converted into a supervised biological negative.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

## 1. Paths

In [2]:
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "EDA" else cwd

PNU_DIR = PROJECT_ROOT / "data" / "processed" / "pnu_universes"

MASTER_PATH = PNU_DIR / "pnu_sequence_master.tsv.gz"
CORE_OVERLAP_PATH = PNU_DIR / "background_known_p_overlap.tsv.gz"

RESULT_DIR = PROJECT_ROOT / "EDA_results" / "E0"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = RESULT_DIR / "phagelytic_eda_universe.csv"

for path in [MASTER_PATH, CORE_OVERLAP_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found: {path}")

print(f"Project root : {PROJECT_ROOT}")
print(f"Master input : {MASTER_PATH}")
print(f"Core overlap : {CORE_OVERLAP_PATH}")
print(f"Output       : {OUTPUT_PATH}")

Project root : /home/david/Desktop/colabs/inta_project/phagelytic_core_project/phagelysclass
Master input : /home/david/Desktop/colabs/inta_project/phagelytic_core_project/phagelysclass/data/processed/pnu_universes/pnu_sequence_master.tsv.gz
Core overlap : /home/david/Desktop/colabs/inta_project/phagelytic_core_project/phagelysclass/data/processed/pnu_universes/background_known_p_overlap.tsv.gz
Output       : /home/david/Desktop/colabs/inta_project/phagelytic_core_project/phagelysclass/EDA_results/E0/phagelytic_eda_universe.csv


## 2. Load frozen sequence-level inputs

In [3]:
master = pd.read_csv(
    MASTER_PATH,
    sep="\t",
    compression="infer",
    dtype=str,
    low_memory=False,
).fillna("")

core_overlap = pd.read_csv(
    CORE_OVERLAP_PATH,
    sep="\t",
    compression="infer",
    dtype=str,
    low_memory=False,
).fillna("")

assert len(master) == 442_329, f"Unexpected master size: {len(master):,}"
assert master["sequence"].ne("").all()
assert master["sequence"].is_unique
assert master["sequence_sha256"].is_unique

print(f"Master exact sequences          : {len(master):,}")
print(f"Core sequences observed UniProt : {core_overlap['sequence'].nunique():,}")

display(master["sequence_state"].value_counts().rename("n").to_frame())

Master exact sequences          : 442,329
Core sequences observed UniProt : 6,985


,n
sequence_state,
U_explicit,228075
U_other_unresolved,132145
N_easy,54711
P_core,11867
challenge_ambiguous_structural,10885
N_hard,4327
challenge_lytic_related,319


## 3. Attach UniProt-side evidence to the exact-sequence master

For non-Core rows, the master already contains the collapsed UniProt evidence states.

For `P_core`, the master intentionally keeps the PhageLysData entity and removes the duplicated
UniProt entity. Therefore, `background_known_p_overlap.tsv.gz` is joined back only to indicate
that the exact Core sequence was also observed in UniProt.

This does not alter the frozen primary label.

In [4]:
overlap_evidence = (
    core_overlap[
        [
            "sequence",
            "observed_evidence_states",
            "observed_primary_rules",
            "observed_rule_groups",
        ]
    ]
    .drop_duplicates("sequence")
    .rename(
        columns={
            "observed_evidence_states": "core_overlap_evidence_states",
            "observed_primary_rules": "core_overlap_primary_rules",
            "observed_rule_groups": "core_overlap_rule_groups",
        }
    )
)

df = master.merge(
    overlap_evidence,
    on="sequence",
    how="left",
    validate="one_to_one",
)

for col in [
    "core_overlap_evidence_states",
    "core_overlap_primary_rules",
    "core_overlap_rule_groups",
]:
    df[col] = df[col].fillna("")

# UniProt evidence attached to each exact sequence.
# - background-derived rows: use evidence already stored in the master
# - P_core rows seen in UniProt: use the known-overlap provenance table
# - P_core rows not seen in UniProt: empty string
df["uniprot_observed_evidence_states"] = np.where(
    df["source_universe"].eq("uniprot_background"),
    df["observed_evidence_states"],
    df["core_overlap_evidence_states"],
)

# True for every exact sequence actually observed in the UniProt phage background.
# This includes all UniProt-derived master rows plus the 6,985 P_core exact overlaps.
core_overlap_sequences = set(core_overlap["sequence"])

df["present_in_uniprot_background"] = (
    df["source_universe"].eq("uniprot_background")
    | df["sequence"].isin(core_overlap_sequences)
)

print(
    "Exact sequences observed in UniProt:",
    f"{int(df['present_in_uniprot_background'].sum()):,}",
)
print(
    "P_core exact overlaps with UniProt:",
    f"{int((df['sequence_state'].eq('P_core') & df['present_in_uniprot_background']).sum()):,}",
)

display(
    df[
        [
            "sequence_state",
            "source_universe",
            "present_in_uniprot_background",
            "uniprot_observed_evidence_states",
        ]
    ].head()
)

Exact sequences observed in UniProt: 437,447
P_core exact overlaps with UniProt: 6,985


,sequence_state,source_universe,present_in_uniprot_background,uniprot_observed_evidence_states
0,P_core,phagelysdata_core,False,
1,P_core,phagelysdata_core,False,
2,P_core,phagelysdata_core,False,
3,P_core,phagelysdata_core,False,
4,P_core,phagelysdata_core,True,known_p_overlap


## 4. Build independent evidence-membership flags

In [5]:
U_EXPLICIT_STATES = {
    "u_uncharacterized",
    "u_domain_unknown",
    "u_low_information",
}

def parse_states(value):
    if value is None or pd.isna(value):
        return set()
    return {item for item in str(value).split(";") if item}

observed_sets = df["uniprot_observed_evidence_states"].map(parse_states)

# Curated positive membership.
df["P_core"] = df["sequence_state"].eq("P_core")

# Independent UniProt evidence flags.
# These are deliberately not mutually exclusive.
df["N_easy"] = (
    observed_sets.map(lambda s: "n_candidate_easy" in s)
    | df["sequence_state"].eq("N_easy")
)

df["N_hard"] = (
    observed_sets.map(lambda s: "n_candidate_hard" in s)
    | df["sequence_state"].eq("N_hard")
)

df["U_explicit"] = (
    observed_sets.map(lambda s: bool(s & U_EXPLICIT_STATES))
    | df["sequence_state"].eq("U_explicit")
)

df["U_other_unresolved"] = (
    observed_sets.map(lambda s: "u_other_unresolved" in s)
    | df["sequence_state"].eq("U_other_unresolved")
)

df["challenge_lytic_related"] = (
    observed_sets.map(lambda s: "lytic_related" in s)
    | df["sequence_state"].eq("challenge_lytic_related")
)

df["challenge_ambiguous_structural"] = (
    observed_sets.map(lambda s: "ambiguous_structural" in s)
    | df["sequence_state"].eq("challenge_ambiguous_structural")
)

membership_cols = [
    "P_core",
    "N_easy",
    "N_hard",
    "U_explicit",
    "U_other_unresolved",
    "challenge_lytic_related",
    "challenge_ambiguous_structural",
]

df["n_memberships"] = df[membership_cols].sum(axis=1).astype(int)
df["multiple_memberships"] = df["n_memberships"].gt(1)

df["membership_pattern"] = df[membership_cols].apply(
    lambda row: ";".join(
        col for col, value in row.items() if bool(value)
    ),
    axis=1,
)

flag_counts = pd.DataFrame(
    {
        "membership": membership_cols,
        "n": [int(df[col].sum()) for col in membership_cols],
    }
)

display(flag_counts)

,membership,n
0,P_core,11867
1,N_easy,54751
2,N_hard,4328
3,U_explicit,229009
4,U_other_unresolved,138745
5,challenge_lytic_related,319
6,challenge_ambiguous_structural,10885


## 5. Add EDA labels

In [6]:
# Conservative biological label for downstream EDA.
# U/challenge sequences remain unknown.
df["phage_lytic"] = "unknown"
df.loc[df["sequence_state"].isin(["N_easy", "N_hard"]), "phage_lytic"] = "no"
df.loc[df["P_core"], "phage_lytic"] = "yes"

df["phage_lytic_subtype"] = ""
df.loc[df["P_core"], "phage_lytic_subtype"] = df.loc[
    df["P_core"],
    "canonical_target_class",
]

CLEAN_SUBTYPES = {
    "endolysin",
    "depolymerase",
    "virion_associated_lytic_enzyme",
}

df["clean_subtype"] = df["phage_lytic_subtype"].where(
    df["phage_lytic_subtype"].isin(CLEAN_SUBTYPES),
    "",
)

df["sequence_length"] = df["sequence"].str.len().astype(int)
df["sequence_uid"] = df["sequence_sha256"]

display(df["phage_lytic"].value_counts().rename("n").to_frame())
display(df["phage_lytic_subtype"].value_counts().rename("n").to_frame())

,n
phage_lytic,
unknown,371424
no,59038
yes,11867


,n
phage_lytic_subtype,
,430462
endolysin,9888
multiple_nonexclusive,999
depolymerase,712
virion_associated_lytic_enzyme,241
other_phage_envelope_lytic_enzyme,27


## 6. Audit primary states and evidence overlaps

In [7]:
primary_counts = (
    df["sequence_state"]
    .value_counts()
    .rename_axis("primary_state")
    .rename("n")
    .reset_index()
)

overlap_summary = (
    df.loc[df["multiple_memberships"], "membership_pattern"]
    .value_counts()
    .rename_axis("membership_pattern")
    .rename("n_sequences")
    .reset_index()
)

pairwise_overlap = pd.DataFrame(
    0,
    index=membership_cols,
    columns=membership_cols,
    dtype=int,
)

for a in membership_cols:
    for b in membership_cols:
        pairwise_overlap.loc[a, b] = int((df[a] & df[b]).sum())

print(f"Sequences with >1 membership flag: {int(df['multiple_memberships'].sum()):,}")
display(primary_counts)
display(overlap_summary.head(25))
display(pairwise_overlap)

Sequences with >1 membership flag: 7,295


,primary_state,n
0,U_explicit,228075
1,U_other_unresolved,132145
2,N_easy,54711
3,P_core,11867
4,challenge_ambiguous_structural,10885
5,N_hard,4327
6,challenge_lytic_related,319


,membership_pattern,n_sequences
0,U_explicit;U_other_unresolved,4373
1,N_easy;U_other_unresolved,1478
2,N_easy;U_explicit,468
3,U_other_unresolved;challenge_ambiguous_structural,325
4,N_easy;U_explicit;U_other_unresolved,179
5,N_hard;U_other_unresolved,135
6,U_explicit;challenge_ambiguous_structural,117
7,N_hard;U_explicit,86
8,U_explicit;U_other_unresolved;challenge_ambigu...,48
9,N_hard;U_explicit;U_other_unresolved,25


,P_core,N_easy,N_hard,U_explicit,U_other_unresolved,challenge_lytic_related,challenge_ambiguous_structural
P_core,11867,0,0,0,0,0,0
N_easy,0,54751,6,653,1675,0,34
N_hard,0,6,4328,115,164,0,1
U_explicit,0,653,115,229009,4631,4,169
U_other_unresolved,0,1675,164,4631,138745,18,389
challenge_lytic_related,0,0,0,4,18,319,0
challenge_ambiguous_structural,0,34,1,169,389,0,10885


## 7. Closure checks

In [8]:
EXPECTED_PRIMARY_COUNTS = {
    "P_core": 11_867,
    "N_easy": 54_711,
    "N_hard": 4_327,
    "U_explicit": 228_075,
    "U_other_unresolved": 132_145,
    "challenge_lytic_related": 319,
    "challenge_ambiguous_structural": 10_885,
}

observed_primary_counts = df["sequence_state"].value_counts().to_dict()

for state, expected in EXPECTED_PRIMARY_COUNTS.items():
    observed = int(observed_primary_counts.get(state, 0))
    assert observed == expected, (
        f"Primary count mismatch for {state}: "
        f"{observed:,} != {expected:,}"
    )

assert len(df) == 442_329
assert df["sequence"].is_unique
assert df["sequence_uid"].is_unique

assert int(df["present_in_uniprot_background"].sum()) == 437_447
assert int((df["P_core"] & df["present_in_uniprot_background"]).sum()) == 6_985

assert int(df["P_core"].sum()) == 11_867
assert int((df["sequence_state"].eq("N_easy")).sum()) == 54_711
assert int((df["sequence_state"].eq("N_hard")).sum()) == 4_327

assert set(df["phage_lytic"].unique()) <= {"yes", "no", "unknown"}
assert df.loc[df["phage_lytic"].eq("yes"), "P_core"].all()
assert df.loc[df["phage_lytic"].eq("no"), "sequence_state"].isin(
    ["N_easy", "N_hard"]
).all()

print("PASS — E0 universe and evidence pivot are internally consistent.")

PASS — E0 universe and evidence pivot are internally consistent.


## 8. Freeze E0

In [9]:
output_columns = [
    "sequence_uid",
    "sequence",
    "sequence_length",
    "phage_lytic",
    "phage_lytic_subtype",
    "clean_subtype",
    "P_core",
    "N_easy",
    "N_hard",
    "U_explicit",
    "U_other_unresolved",
    "challenge_lytic_related",
    "challenge_ambiguous_structural",
    "n_memberships",
    "multiple_memberships",
    "membership_pattern",
    "sequence_state",
    "pnu_role",
    "stratum",
    "assigned_tier",
    "present_in_uniprot_background",
    "uniprot_observed_evidence_states",
]

eda = df[output_columns].copy()

eda.to_csv(OUTPUT_PATH, index=False)

print(f"Saved : {OUTPUT_PATH}")
print(f"Rows  : {len(eda):,}")
print(f"Cols  : {eda.shape[1]}")
print(f"Size  : {OUTPUT_PATH.stat().st_size / 1024**2:.2f} MB")

display(eda.head())

Saved : /home/david/Desktop/colabs/inta_project/phagelytic_core_project/phagelysclass/EDA_results/E0/phagelytic_eda_universe.csv
Rows  : 442,329
Cols  : 22
Size  : 179.39 MB


,sequence_uid,sequence,sequence_length,phage_lytic,phage_lytic_subtype,clean_subtype,P_core,N_easy,N_hard,U_explicit,U_other_unresolved,challenge_lytic_related,challenge_ambiguous_structural,n_memberships,multiple_memberships,membership_pattern,sequence_state,pnu_role,stratum,assigned_tier,present_in_uniprot_background,uniprot_observed_evidence_states
0,0009e722fb00b8c7fc49e396d793ae670960d349e64559...,MTEKVLPFDRKIIRQDTGYWCGPASAQVALSARGKYVDEATLAREC...,519,yes,endolysin,endolysin,True,False,False,False,False,False,False,1,False,P_core,P_core,P,endolysin,Tier 3,False,
1,00122e3566692ed6ac32a7206440900d95a80aaccfcc50...,MREAYLILVPGTSETHDAPSPLPPTGMTKLISSALGRLMETDHEWA...,285,yes,endolysin,endolysin,True,False,False,False,False,False,False,1,False,P_core,P_core,P,endolysin,Tier 3,False,
2,00169670683d5c47b0d5dbe58746198da271112addae42...,MLIQRSAALGAAFLVVTLLAVLCAPQGWVPAAAARSATDPVERLRR...,350,yes,endolysin,endolysin,True,False,False,False,False,False,False,1,False,P_core,P_core,P,endolysin,Tier 3,False,
3,0019671f9f9b71056d5a91dfe72a8a1725d84b316bf5d8...,MILKKGSNGALVRLWTNVMLSRFKSYALGVDGKPLVNDGYFGNDEE...,330,yes,endolysin,endolysin,True,False,False,False,False,False,False,1,False,P_core,P_core,P,endolysin,Tier 3,False,
4,002209216b60a0f7d07bad61752c8311ff2c6d5ba9943b...,MAKVQFNQRTKTEAIFIHCSATKPSMDIGVREIRQWHKEQGWLDVG...,151,yes,endolysin,endolysin,True,False,False,False,False,False,False,1,False,P_core,P_core,P,endolysin,Tier 3,True,known_p_overlap
